## Objetivo

Realizar a transformação e padronização dos dados provenientes da camada Bronze, preparando-os para análises exploratórias,
indicadores de desempenho e análises de investimento em imóveis para aluguel de curta duração em João Pessoa - PB.

## Fonte dos dados

Os dados utilizados neste notebook são provenientes das tabelas Delta da camada Bronze:

- `airbnb_joao_pessoa.bronze.listings`
- `airbnb_joao_pessoa.bronze.past_calendar_rates`
- `airbnb_joao_pessoa.bronze.future_calendar_rates`
- `airbnb_joao_pessoa.bronze.reviews`

## Responsabilidades da camada Silver

Nesta camada serão realizadas atividades de:

- avaliação da qualidade dos dados;
- identificação e tratamento de valores nulos;
- identificação e tratamento de registros duplicados;
- validação de tipos e domínios dos dados;
- padronização de campos;
- tratamento de valores inconsistentes ou inválidos;
- criação de atributos derivados quando necessário;
- preparação dos dados para as análises da camada Gold.

## Princípio de transformação

A camada Silver não tem como objetivo alterar a informação original sem justificativa. As transformações serão baseadas em regras de qualidade e nas necessidades analíticas identificadas durante a investigação dos dados.

Os dados originais são preservados na camada Bronze.

## Tabelas de saída

Ao final do processo, serão disponibilizadas as seguintes tabelas na camada Silver:

- `airbnb_joao_pessoa.silver.listings`
- `airbnb_joao_pessoa.silver.past_calendar_rates`
- `airbnb_joao_pessoa.silver.future_calendar_rates`
- `airbnb_joao_pessoa.silver.reviews`

In [0]:
catalog = "airbnb_joao_pessoa"
bronze_schema = "bronze"
raw_path = f"/Volumes/{catalog}/{bronze_schema}/raw_files"

In [0]:
listings = spark.table("airbnb_joao_pessoa.bronze.listings")
past_rates = spark.table("airbnb_joao_pessoa.bronze.past_calendar_rates")
future_rates = spark.table("airbnb_joao_pessoa.bronze.future_calendar_rates")
reviews = spark.table("airbnb_joao_pessoa.bronze.reviews")

### Auditoria de qualidade

In [0]:
from pyspark.sql.functions import col, sum, when

dataframes = {
    "listings": listings,
    "past_rates": past_rates,
    "future_rater": future_rates,
    "reviews": reviews
}

In [0]:
# Nulos

for nome, df in dataframes.items():
    print(f"Nulos em: {nome}")
    display(
        df.select([
            sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in df.columns
        ])
    )

In [0]:
# Duplicados
for nome, df in dataframes.items():
    print(f"Duplicidade em: {nome}")
    df.groupBy("listing_id").count().filter(col("count") > 1).show() 

In [0]:
# Auditoria de qualidade

display(
    listings.select(
        "listing_id",
        "listing_type",
        "room_type",
        "guests",
        "bedrooms",
        "beds",
        "baths",
        "rating_overall",
        "ttm_revenue",
        "ttm_occupancy"
    )
)

In [0]:
display(
    past_rates.select(
        "listing_id",
        "date",
        "vacant_days",
        "reserved_days",
        "occupancy",
        "revenue",
        "rate_avg",
        "booked_rate_avg",
        "booking_lead_time_avg",
        "length_of_stay_avg",
        "min_nights_avg",
        "native_booked_rate_avg",
        "native_rate_avg",
        "native_revenue"
    )
)

In [0]:
display(
    future_rates.select(
        "listing_id",
        "date",
        "vacant_days",
        "reserved_days",
        "occupancy",
        "revenue",
        "rate_avg",
        "booked_rate_avg",
        "booking_lead_time_avg",
        "length_of_stay_avg",
        "min_nights_avg",
        "native_booked_rate_avg",
        "native_rate_avg",
        "native_revenue"
    )
)

In [0]:
display(
    reviews.select(
        "listing_id",
        "date",
        "num_reviews",
        "reviewers"
    )
)